# OPALX run analysis

Three things:

1. **Stat browser** — pick runs and `.stat` columns, plot them against `s` and/or `t`.
2. **Phase-space heatmaps** — 2D histograms of the monitor planes and the bunch dumps.
3. **ParaView export** — bunch particles per step in both the co-moving and the lab frame,
   plus the lattice aperture tubes and the reference orbit.

Nothing here is specific to the aperture study. `ROOT` in the next cell names a directory;
every run under it is found as `<case>/output/*.stat` (or `<case>/*.stat`). Point `ROOT` at
`../bend` and this reads the bend runs instead.

Run it with the miniconda python — `/opt/homebrew/Caskroom/miniconda/base/bin/python`, the
`python3` kernel. The system python has neither `h5py` nor `vtk`.

In [1]:
from pathlib import Path
import re
from functools import lru_cache

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Ellipse, Rectangle
import h5py
import ipywidgets as widgets
from IPython.display import display

# ------------------------------------------------------------------ settings --
# The only line to change to look at a different study.
ROOT = Path(".").resolve()

# Rest mass for the Ekin array in the ParaView export.
# The bunch .h5 does carry a MASS step attribute, but it reads 5.103 for an
# electron whose mass is 0.511 MeV -- a factor 10 out -- so it is not used.
# load_cases() cross-checks this constant against the value implied by the
# ENERGY and RefPartP step attributes and warns if they disagree.
REST_MASS_MEV = 0.51099895            # electron; muon = 105.6583755

# Radius [m] drawn for elements whose deck line declares no APERTURE. Only
# affects how the ParaView lattice looks, never the physics.
DEFAULT_APERTURE_M = 0.02

plt.rcParams.update({
    "figure.figsize": (11, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "axes.axisbelow": True,
    "font.size": 10,
})

CASE_COLORS = ["#1f5fd1", "#c1432d", "#1e8a4c", "#8a5cc1", "#c98a1e", "#0f8b8d"]

---
## Library

Readers, frame math and VTK writers. **You should not need to read this cell.**

It is a self-contained copy of the parts of `runs/processing/{opalx_diagnostics, opalx_run,
particles_to_vtk, elements_to_vtk}.py` that this notebook uses, so the notebook works with
nothing outside `opalx-runs` on the path.

In [2]:
# ============================================================================ #
#  Library -- skip this cell.                                                  #
# ============================================================================ #

# ------------------------------------------------------------- run discovery --
def discover_cases(root):
    """Every OPALX run under `root`, as {case_name: info dict}.

    A run is a directory holding a `.stat`. Output may sit in a nested
    `output/` folder or directly in the case directory; both are found.
    The deck is looked for in the case directory, i.e. one level above
    `output/` when that layout is used.
    """
    root = Path(root).resolve()
    found = {}
    for stat in sorted(root.glob("**/*.stat")):
        out_dir = stat.parent
        case_dir = out_dir.parent if out_dir.name == "output" else out_dir
        name = case_dir.name
        base = re.sub(r"_c\d+$", "", stat.stem)
        info = found.get(name)
        if info is None:
            deck = case_dir / f"{base}.in"
            if not deck.is_file():
                decks = sorted(case_dir.glob("*.in"))
                deck = decks[0] if decks else None
            bunch = out_dir / f"{base}.h5"
            info = {
                "name": name,
                "dir": case_dir,
                "out": out_dir,
                "base": base,
                "deck": deck,
                "stat": stat,
                "stats": [],
                "bunch_h5": bunch if bunch.is_file() else None,
                "monitors": sorted(p for p in out_dir.glob("*.h5") if p != bunch),
                "data": (out_dir / "data") if (out_dir / "data").is_dir() else None,
            }
            found[name] = info
        info["stats"].append(stat)
    for info in found.values():
        d = info["data"]
        info["designpath"] = next(iter(sorted(d.glob("*_DesignPath.dat"))), None) if d else None
        info["elempos"] = next(iter(sorted(d.glob("*_ElementPositions.txt"))), None) if d else None
    return found


def inventory(cases):
    """One row per run, so a wrong ROOT shows up as an empty table."""
    rows = []
    for name, c in cases.items():
        rows.append({
            "case": name,
            "deck": c["deck"].name if c["deck"] else "-",
            "stat": len(c["stats"]),
            "bunch.h5": c["bunch_h5"].name if c["bunch_h5"] else "-",
            "monitors": ", ".join(p.stem for p in c["monitors"]) or "-",
            "designpath": "yes" if c["designpath"] else "-",
            "elempos": "yes" if c["elempos"] else "-",
        })
    return pd.DataFrame(rows).set_index("case") if rows else pd.DataFrame()


# ------------------------------------------------------------- .stat reading --
def parse_opal_stat(path):
    """Parse an OPALX SDDS `.stat`. Returns (metadata, DataFrame, units)."""
    lines = Path(path).read_text().splitlines()
    params, cols, units = [], [], {}

    def block(i):
        """Collect `name=` / `units=` from one &parameter or &column block."""
        name = unit = None
        i += 1
        while i < len(lines) and lines[i].strip() != "&end":
            m = re.search(r"name\s*=\s*([^,]+)", lines[i])
            if m:
                name = m.group(1).strip()
            m = re.search(r"units\s*=\s*([^,]+)", lines[i])
            if m:
                unit = m.group(1).strip()
            i += 1
        return i, name, unit

    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line == "&parameter":
            i, name, _ = block(i)
            if name:
                params.append(name)
        elif line == "&column":
            i, name, unit = block(i)
            if name:
                cols.append(name)
                units[name] = unit
        elif line == "&data":
            while i < len(lines) and lines[i].strip() != "&end":
                i += 1
            i += 1
            break
        i += 1

    body = [ln.strip() for ln in lines[i:] if ln.strip()]
    meta = dict(zip(params, body[: len(params)]))
    rows = []
    for ln in body[len(params):]:
        tok = ln.split()
        if len(tok) < len(cols):
            continue
        rows.append([np.nan if t.lower() == "nan" else float(t) for t in tok[: len(cols)]])
    return meta, pd.DataFrame(rows, columns=cols), units


@lru_cache(maxsize=64)
def _stat_cached(path_str):
    return parse_opal_stat(path_str)


def stat_of(case_info):
    """(metadata, df, units) for a case, cached by path."""
    return _stat_cached(str(case_info["stat"]))


# --------------------------------------------------------------- .h5 reading --
_PARTICLE_KEYS = ("x", "y", "z", "px", "py", "pz")


def h5_steps(path):
    with h5py.File(path, "r") as f:
        return sorted(int(k.split("#")[1]) for k in f if k.startswith("Step#"))


def _pack(group):
    n = len(group["x"])
    d = {k: np.asarray(group[k][()]).reshape(-1) for k in _PARTICLE_KEYS if k in group}
    d["ids"] = np.asarray(group["id"][()]).reshape(-1) if "id" in group else np.arange(n)
    spos = group.attrs.get("SPOS")
    d["spos"] = float(np.ravel(spos)[0]) if spos is not None else float("nan")
    t = group.attrs.get("TIME")
    d["time"] = float(np.ravel(t)[0]) if t is not None else float("nan")
    return d


def read_monitor(path):
    """All Step#N groups of a monitor file, concatenated.

    A single pass through a monitor is written as SEVERAL Step groups --
    LossDataSink::splitSets partitions it in time -- so reading Step#0 alone
    under-reports the crossing. Duplicate ids mean a particle crossed twice;
    that is reported, never silently deduplicated.
    """
    parts, spos, warn = [], [], []
    with h5py.File(path, "r") as f:
        steps = sorted(int(k.split("#")[1]) for k in f if k.startswith("Step#"))
        for n in steps:
            g = f[f"Step#{n}"]
            if "x" not in g or len(g["x"]) == 0:
                continue
            d = _pack(g)
            parts.append(d)
            if np.isfinite(d["spos"]):
                spos.append(d["spos"])
    if not parts:
        return None
    out = {k: np.concatenate([p[k] for p in parts if k in p])
           for k in list(_PARTICLE_KEYS) + ["ids"] if k in parts[0]}
    out["spos"] = float(np.mean(spos)) if spos else float("nan")
    out["nsets"] = len(parts)
    uniq, counts = np.unique(out["ids"], return_counts=True)
    if len(uniq) != len(out["ids"]):
        warn.append(f"{Path(path).name}: {len(out['ids']) - len(uniq)} duplicate id(s) "
                    f"across {len(parts)} Step groups (a second crossing?)")
    out["warnings"] = warn
    return out


def read_step(path, step):
    """One Step#N group -- what you want for a bunch dump, where each group
    is a different time rather than a slice of one crossing."""
    with h5py.File(path, "r") as f:
        key = f"Step#{int(step)}"
        if key not in f or len(f[key].get("x", [])) == 0:
            return None
        d = _pack(f[key])
    d["nsets"] = 1
    d["warnings"] = []
    return d


def geom_emittance(u, up):
    """sqrt(<u^2><u'^2> - <u u'>^2), mean subtracted."""
    u = u - u.mean()
    up = up - up.mean()
    return float(np.sqrt(max(np.mean(u * u) * np.mean(up * up) - np.mean(u * up) ** 2, 0.0)))


# ----------------------------------------------------- deck: element apertures --
ELEMENT_TYPES = {
    "DRIFT", "MONITOR", "QUADRUPOLE", "SEXTUPOLE", "OCTUPOLE", "MULTIPOLE", "MULTIPOLET",
    "SOLENOID", "SBEND", "RBEND", "RFCAVITY", "TRAVELINGWAVE", "PROBE", "MARKER",
    "COLLIMATOR", "ECOLLIMATOR", "RCOLLIMATOR", "VACUUM", "SOURCE", "LASER",
    "CONSTANTEFIELDCAVITY", "VERTICALFFAMAGNET", "VARIABLERFCAVITY",
}
TYPE_CODE = {"DRIFT": 0, "SOLENOID": 1, "SBEND": 2, "RBEND": 2, "QUADRUPOLE": 3, "MONITOR": 4}
TYPE_LEGEND = {0: "DRIFT", 1: "SOLENOID", 2: "DIPOLE", 3: "QUADRUPOLE", 4: "MONITOR", 5: "OTHER"}


def strip_comments(text):
    text = re.sub(r"/\*.*?\*/", "", text, flags=re.DOTALL)
    return re.sub(r"//[^\n]*", "", text)


def parse_reals(text):
    """`REAL name = value;` assignments, for decks that name their apertures."""
    syms = {}
    for m in re.finditer(r"\bREAL\s+(\w+)\s*=\s*([^;]+);", text):
        try:
            syms[m.group(1)] = float(m.group(2))
        except ValueError:
            pass
    return syms


def _num(token, syms):
    token = token.strip()
    try:
        return float(token)
    except ValueError:
        return syms.get(token)


def parse_aperture_string(s):
    """`'ELLIPSE(0.02, 0.01)'` -> `('ell', 0.01, 0.005)`.

    Deck arguments are FULL widths / diameters and are halved here, matching
    OpalElement::getApert (width2HalfWidth = 0.5).
    """
    m = re.search(r'"?\s*(square|rectangle|circle|ellipse)\s*\(([^)]*)\)', s, re.IGNORECASE)
    if not m:
        return None
    shape = m.group(1).lower()
    args = [float(a) for a in re.split(r"[,\s]+", m.group(2).strip()) if a]
    if shape == "rectangle":
        return ("rect", args[0] / 2, args[1] / 2)
    if shape == "square":
        return ("rect", args[0] / 2, args[0] / 2)
    if shape == "ellipse":
        return ("ell", args[0] / 2, args[1] / 2)
    if shape == "circle":
        return ("ell", args[0] / 2, args[0] / 2)
    return None


def parse_deck(deck_path):
    """{name: {"type", "aper"}} for every element the deck declares.

    `aper` is None when the deck declares none -- callers decide whether to
    substitute a placeholder. Bends carry HAPERT/HGAP instead of APERTURE, and
    those are already half-values, so they are not halved.
    """
    text = strip_comments(Path(deck_path).read_text())
    syms = parse_reals(text)
    elements = {}
    for stmt in text.split(";"):
        m = re.match(r"\s*(\w+)\s*:\s*(\w+)\b(.*)", stmt, flags=re.DOTALL)
        if not m:
            continue
        name, typ, rest = m.group(1), m.group(2).upper(), m.group(3)
        if typ not in ELEMENT_TYPES:
            continue
        aper = None
        if typ in ("SBEND", "RBEND"):
            hap = re.search(r"\bHAPERT\s*=\s*([^\s,]+)", rest)
            hg = re.search(r"\bHGAP\s*=\s*([^\s,]+)", rest)
            hx = _num(hap.group(1), syms) if hap else None
            hy = _num(hg.group(1), syms) if hg else None
            if hx is not None:
                aper = ("rect", hx, hy if hy is not None else hx)
        else:
            # The value contains a comma (`ELLIPSE(0.02, 0.01)`), so a plain
            # "up to the next comma" match would cut it in half. Take a quoted
            # string whole, or a NAME(...) group whole, before falling back.
            am = re.search(r'\bAPERTURE\s*=\s*("[^"]*"|\w+\s*\([^)]*\)|[^,;]*)', rest)
            if am:
                aper = parse_aperture_string(am.group(1))
        elements[name] = {"type": typ, "aper": aper}
    return elements


def declared_apertures(case_info):
    """Only the elements whose deck line actually sets an aperture."""
    if not case_info.get("deck"):
        return {}
    return {n: e["aper"] for n, e in parse_deck(case_info["deck"]).items() if e["aper"]}


# ------------------------------------------------- design path / element paths --
ZHAT = np.array([0.0, 0.0, 1.0])


def load_designpath(path):
    """Reference orbit as (s, R, P), sorted by path length.
    Columns are `s Rx Ry Rz Px Py Pz ...`; R is lab position [m], P is beta*gamma."""
    s, R, P = [], [], []
    for line in Path(path).read_text().splitlines():
        if line.startswith("#"):
            continue
        f = line.split()
        if len(f) < 7:
            continue
        try:
            s.append(float(f[0]))
            R.append([float(f[1]), float(f[2]), float(f[3])])
            P.append([float(f[4]), float(f[5]), float(f[6])])
        except ValueError:
            continue
    s, R, P = np.asarray(s), np.asarray(R), np.asarray(P)
    o = np.argsort(s)
    return s[o], R[o], P[o]


_ELEMPOS_ROW = re.compile(r'"([^:]+):\s*(\w+)"\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s+([-\d.eE+]+)')


def parse_element_positions(path):
    """Per-element lab centerlines as [(name, (N,3) points)].

    OPALX writes BEGIN / MID / END (and ENTRY/EXIT EDGE for bends) rows, so a
    bent dipole already comes out as an arc -- no placement or bend math is
    redone. File column order is Z X Y.
    """
    order, pts = [], {}
    for line in Path(path).read_text().splitlines():
        m = _ELEMPOS_ROW.match(line)
        if not m:
            continue
        name = m.group(2)
        c1, c2, c3 = float(m.group(3)), float(m.group(4)), float(m.group(5))
        if name not in pts:
            pts[name] = []
            order.append(name)
        pts[name].append((c2, c3, c1))          # Z X Y -> x y z
    out = []
    for name in order:
        P = np.asarray(pts[name], dtype=float)
        keep = [0]
        for i in range(1, len(P)):
            if np.linalg.norm(P[i] - P[keep[-1]]) > 1e-9:
                keep.append(i)
        out.append((name, P[keep]))
    return out


# ------------------------------------------------------- co-moving -> lab frame --
def shortest_arc(a, b):
    """Rotation taking unit vector `a` onto `b` with no roll."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    a = a / np.linalg.norm(a); b = b / np.linalg.norm(b)
    v = np.cross(a, b)
    c = float(np.dot(a, b))
    s = np.linalg.norm(v)
    if s < 1e-12:
        return np.eye(3) if c > 0 else np.diag([1.0, -1.0, -1.0])
    vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    return np.eye(3) + vx + vx @ vx * ((1.0 - c) / (s * s))


def build_transport_frames(dhat):
    """Co-moving -> lab rotation Q(k) at every reference-orbit sample.

    OPALX realigns its frame each step by the shortest-arc rotation putting
    local +z on the new reference momentum. Composing those increments is
    parallel transport, which reproduces OPALX's own toLabTrafo including the
    accumulated roll that a single shortest-arc from +z would miss.
    """
    dhat = dhat / np.linalg.norm(dhat, axis=1, keepdims=True)
    Q = np.empty((len(dhat), 3, 3))
    Q[0] = shortest_arc(ZHAT, dhat[0])
    for k in range(1, len(dhat)):
        Q[k] = shortest_arc(dhat[k - 1], dhat[k]) @ Q[k - 1]
    return Q


def lab_frames(case_info):
    """(s_samples, Q_samples) for the lab transform, or None if no DesignPath."""
    dp = case_info.get("designpath")
    if dp is None:
        return None
    s_dp, _, p_dp = load_designpath(dp)
    return s_dp, build_transport_frames(p_dp)


def check_rest_mass(case_info):
    """Rest mass [MeV] implied by the bunch step attrs, or None.

    Ekin = m (gamma - 1) with gamma = sqrt(1 + |p|^2), p in beta*gamma. Used
    only to sanity-check REST_MASS_MEV; the MASS step attribute itself is not
    trusted (it reads 10x the electron mass).
    """
    if not case_info.get("bunch_h5"):
        return None
    with h5py.File(case_info["bunch_h5"], "r") as f:
        keys = [k for k in f if k.startswith("Step#")]
        if not keys:
            return None
        g = f[sorted(keys, key=lambda k: int(k.split("#")[1]))[0]]
        ekin = float(np.ravel(g.attrs["ENERGY"])[0])                 # MeV
        bg = float(np.linalg.norm(np.ravel(g.attrs["RefPartP"])[:3]))
    gamma = np.sqrt(1.0 + bg * bg)
    return ekin / (gamma - 1.0) if gamma > 1.0 else None


# ------------------------------------------------------------------ VTK output --
VECTOR_TRIPLES = {"Pol": ("polx", "poly", "polz"),
                  "E": ("Ex", "Ey", "Ez"),
                  "B": ("Bx", "By", "Bz")}
_TRIPLE_MEMBERS = {n for t in VECTOR_TRIPLES.values() for n in t}
_HANDLED = set(_PARTICLE_KEYS) | _TRIPLE_MEMBERS


def _vtk():
    """Imported lazily so the plotting sections work without vtk installed."""
    import vtk
    from vtk.util.numpy_support import numpy_to_vtk
    return vtk, numpy_to_vtk


def _to_1d(group, name):
    return np.asarray(group[name][()]).reshape(-1)


def _extra_arrays(group, n):
    """Everything not already handled, as (name, values, is_vector).

    Known triples become (n,3) vectors so they can be rotated with the frame;
    any other 1-D per-particle dataset is passed through as a scalar, so a run
    carrying datasets this notebook has never heard of still exports them.
    """
    out = []
    for name, cols in VECTOR_TRIPLES.items():
        if all(c in group for c in cols):
            vec = np.column_stack([_to_1d(group, c) for c in cols])
            if len(vec) == n:
                out.append((name, vec, True))
    for name in sorted(group):
        if name in _HANDLED or getattr(group[name], "ndim", 0) != 1:
            continue
        vals = _to_1d(group, name)
        if len(vals) == n:
            out.append((name, vals.astype(np.float64), False))
    return out


def _write_step_vtp(group, rot, origin, mass_mev, out_file):
    """One .vtp. `rot`/`origin` are None for the untransformed (co-moving) frame.
    Returns (n_particles, time)."""
    vtk, numpy_to_vtk = _vtk()

    def add(poly, name, values):
        arr = numpy_to_vtk(np.ascontiguousarray(values, dtype=np.float64), deep=1)
        arr.SetName(name)
        poly.GetPointData().AddArray(arr)

    r_local = np.column_stack([_to_1d(group, k) for k in ("x", "y", "z")])
    p_local = np.column_stack([_to_1d(group, k) for k in ("px", "py", "pz")])
    n = len(r_local)
    if rot is None:
        r_out, p_out = r_local, p_local
    else:
        r_out = origin + r_local @ rot.T
        p_out = p_local @ rot.T

    points = vtk.vtkPoints()
    points.SetData(numpy_to_vtk(np.ascontiguousarray(r_out, dtype=np.float64), deep=1))
    verts = vtk.vtkCellArray()
    for i in range(n):
        verts.InsertNextCell(1)
        verts.InsertCellPoint(i)
    poly = vtk.vtkPolyData()
    poly.SetPoints(points)
    poly.SetVerts(verts)

    add(poly, "P", p_out)                              # momentum vector, for glyphs
    bg = np.linalg.norm(p_local, axis=1)               # |p| is frame invariant
    add(poly, "Pmag", bg)                              # beta*gamma
    add(poly, "Ekin_MeV", mass_mev * (np.sqrt(1.0 + bg * bg) - 1.0))
    for name, values, is_vector in _extra_arrays(group, n):
        add(poly, name, values @ rot.T if (is_vector and rot is not None) else values)

    w = vtk.vtkXMLPolyDataWriter()
    w.SetFileName(str(out_file))
    w.SetInputData(poly)
    w.SetDataModeToBinary()
    if w.Write() != 1:
        raise RuntimeError(f"failed writing {out_file}")
    t = group.attrs.get("TIME")
    return n, (float(np.ravel(t)[0]) if t is not None else 0.0)


def _write_pvd(pvd_path, entries):
    lines = ['<?xml version="1.0"?>',
             '<VTKFile type="Collection" version="0.1" byte_order="LittleEndian">',
             "  <Collection>"]
    lines += [f'    <DataSet timestep="{t:.16g}" part="0" file="{f}"/>' for t, f in entries]
    lines += ["  </Collection>", "</VTKFile>", ""]
    Path(pvd_path).write_text("\n".join(lines), encoding="utf-8")


def export_bunch(case_info, frame, out_dir, stride=1, mass_mev=None, log=print):
    """Bunch dumps -> `bunch_<frame>.pvd` + one .vtp per step. Returns the .pvd."""
    h5_path = case_info["bunch_h5"]
    if h5_path is None:
        log(f"  {case_info['name']}: no bunch .h5, skipped")
        return None
    mass_mev = REST_MASS_MEV if mass_mev is None else mass_mev
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    transport = lab_frames(case_info) if frame == "lab" else None
    if frame == "lab" and transport is None:
        log("  WARNING: no *_DesignPath.dat -> falling back to a per-step "
            "shortest-arc rotation (misses frame roll)")

    steps = h5_steps(h5_path)[::stride]
    entries = []
    with h5py.File(h5_path, "r") as f:
        for n in steps:
            g = f[f"Step#{n}"]
            if frame == "lab":
                origin = np.ravel(g.attrs["RefPartR"])[:3]
                if transport is not None:
                    s_dp, Qs = transport
                    spos = float(np.ravel(g.attrs["SPOS"])[0])
                    rot = Qs[int(np.argmin(np.abs(s_dp - spos)))]
                else:
                    rot = shortest_arc(ZHAT, np.ravel(g.attrs["RefPartP"])[:3])
            else:
                origin = rot = None
            fname = f"bunch_{frame}_step{n:05d}.vtp"
            npart, t = _write_step_vtp(g, rot, origin, mass_mev, out_dir / fname)
            entries.append((t, fname))
    pvd = out_dir / f"bunch_{frame}.pvd"
    _write_pvd(pvd, entries)
    log(f"  {pvd.name}  ({len(entries)} steps, {npart} particles in the last)")
    return pvd


def _frames_along(P):
    """Unit tangent, right and up at every polyline vertex (elements are planar)."""
    T = np.empty_like(P)
    T[1:-1] = P[2:] - P[:-2]
    T[0] = P[1] - P[0]
    T[-1] = P[-1] - P[-2]
    T /= np.linalg.norm(T, axis=1, keepdims=True)
    right = np.cross(np.broadcast_to(np.array([0.0, 1.0, 0.0]), P.shape), T)
    bad = np.linalg.norm(right, axis=1) < 1e-8            # tangent nearly vertical
    if bad.any():
        right[bad] = np.cross(np.array([1.0, 0.0, 0.0]), T[bad])
    right /= np.linalg.norm(right, axis=1, keepdims=True)
    up = np.cross(T, right)
    up /= np.linalg.norm(up, axis=1, keepdims=True)
    return right, up


def _cross_section(aper, n_ell):
    shape, hx, hy = aper
    if shape == "rect":
        return [(hx, hy), (-hx, hy), (-hx, -hy), (hx, -hy)]
    th = np.linspace(0.0, 2.0 * np.pi, n_ell, endpoint=False)
    return list(zip(hx * np.cos(th), hy * np.sin(th)))


def _build_tube(P, aper, n_ell, verts, tris, codes, code):
    """Append one element's tube (rings, walls, end caps) to the buffers."""
    right, up = _frames_along(P)
    sec = _cross_section(aper, n_ell)
    m = len(sec)
    base = len(verts)
    for i in range(len(P)):
        for (a, b) in sec:
            verts.append(P[i] + a * right[i] + b * up[i])
    for i in range(len(P) - 1):
        r0, r1 = base + i * m, base + (i + 1) * m
        for j in range(m):
            k = (j + 1) % m
            tris.append((r0 + j, r0 + k, r1 + k)); codes.append(code)
            tris.append((r0 + j, r1 + k, r1 + j)); codes.append(code)
    for ring, P_end in ((base, P[0]), (base + (len(P) - 1) * m, P[-1])):
        c = len(verts)
        verts.append(P_end)
        for j in range(m):
            tris.append((c, ring + j, ring + (j + 1) % m)); codes.append(code)


def _write_triangles(path, verts, tris, codes):
    vtk, numpy_to_vtk = _vtk()
    pts = vtk.vtkPoints()
    pts.SetData(numpy_to_vtk(np.ascontiguousarray(verts, dtype=np.float64), deep=1))
    cells = vtk.vtkCellArray()
    for (a, b, c) in tris:
        t = vtk.vtkTriangle()
        t.GetPointIds().SetId(0, a); t.GetPointIds().SetId(1, b); t.GetPointIds().SetId(2, c)
        cells.InsertNextCell(t)
    poly = vtk.vtkPolyData()
    poly.SetPoints(pts)
    poly.SetPolys(cells)
    arr = numpy_to_vtk(np.ascontiguousarray(codes, dtype=np.int32), deep=1)
    arr.SetName("element_type")
    poly.GetCellData().AddArray(arr)
    w = vtk.vtkXMLPolyDataWriter()
    w.SetFileName(str(path)); w.SetInputData(poly); w.SetDataModeToBinary()
    if w.Write() != 1:
        raise RuntimeError(f"failed writing {path}")


def _write_polyline(path, P):
    vtk, numpy_to_vtk = _vtk()
    pts = vtk.vtkPoints()
    pts.SetData(numpy_to_vtk(np.ascontiguousarray(P, dtype=np.float64), deep=1))
    line = vtk.vtkPolyLine()
    line.GetPointIds().SetNumberOfIds(len(P))
    for i in range(len(P)):
        line.GetPointIds().SetId(i, i)
    cells = vtk.vtkCellArray()
    cells.InsertNextCell(line)
    poly = vtk.vtkPolyData()
    poly.SetPoints(pts); poly.SetLines(cells)
    w = vtk.vtkXMLPolyDataWriter()
    w.SetFileName(str(path)); w.SetInputData(poly); w.SetDataModeToBinary()
    if w.Write() != 1:
        raise RuntimeError(f"failed writing {path}")


def export_geometry(case_info, out_dir, n_ell=24, orbit_stride=10, log=print):
    """Aperture tubes + reference orbit, in the same lab frame as bunch_lab."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    written = []

    if case_info.get("elempos") and case_info.get("deck"):
        elements = parse_deck(case_info["deck"])
        verts, tris, codes = [], [], []
        counts = {}
        for name, P in parse_element_positions(case_info["elempos"]):
            if len(P) < 2:
                continue
            info = elements.get(name, {"type": "OTHER", "aper": None})
            aper = info["aper"] or ("ell", DEFAULT_APERTURE_M, DEFAULT_APERTURE_M)
            code = TYPE_CODE.get(info["type"], 5)
            _build_tube(P, aper, n_ell, verts, tris, codes, code)
            counts[TYPE_LEGEND[code]] = counts.get(TYPE_LEGEND[code], 0) + 1
        if tris:
            p = out_dir / "lattice_elements.vtp"
            _write_triangles(p, np.array(verts), tris, codes)
            written.append(p)
            log(f"  {p.name}  ({len(tris)} triangles; "
                + ", ".join(f"{k}:{v}" for k, v in sorted(counts.items())) + ")")
    else:
        log("  no *_ElementPositions.txt or no deck -> skipped the element tubes")

    if case_info.get("designpath"):
        _, R, _ = load_designpath(case_info["designpath"])
        if orbit_stride > 1:
            R = R[::orbit_stride]
        p = out_dir / "lattice_reforbit.vtp"
        _write_polyline(p, R)
        written.append(p)
        log(f"  {p.name}  ({len(R)} points)")
    else:
        log("  no *_DesignPath.dat -> skipped the reference orbit")
    return written


def export_paraview(case_info, frames=("lab", "comoving"), stride=1, geometry=True, log=print):
    """Everything ParaView needs for one run, into <run>/paraview/."""
    out_dir = Path(case_info["out"]) / "paraview"
    log(f"{case_info['name']} -> {out_dir}")
    made = []
    for frame in frames:
        p = export_bunch(case_info, frame, out_dir, stride=stride, log=log)
        if p:
            made.append(p)
    if geometry:
        made += export_geometry(case_info, out_dir, log=log)
    return made

---
## Runs found

In [3]:
CASES = discover_cases(ROOT)
print(f"ROOT = {ROOT}")
print(f"{len(CASES)} run(s)\n")
display(inventory(CASES))

# The bunch MASS attribute is not trusted; check the configured mass against the
# one the ENERGY / RefPartP attributes imply.
for name, c in CASES.items():
    implied = check_rest_mass(c)
    if implied is None:
        continue
    tag = "ok" if abs(implied - REST_MASS_MEV) / REST_MASS_MEV < 1e-3 else "MISMATCH"
    print(f"rest mass check  {name:20s} deck implies {implied:.6f} MeV  "
          f"vs REST_MASS_MEV = {REST_MASS_MEV:.6f}  [{tag}]")

ROOT = /Users/rammann/Code/OPALX/opalx-runs/studies/aperture
4 run(s)



,deck,stat,bunch.h5,monitors,designpath,elempos
case,,,,,,
drift_ellipse,drift_ellipse.in,1,drift_ellipse.h5,"MON_IN, MON_OUT",yes,yes
drift_ellipse_np2,drift_ellipse_np2.in,1,drift_ellipse_np2.h5,"MON_IN, MON_OUT",yes,yes
drift_noaperture,drift_noaperture.in,1,drift_noaperture.h5,"MON_IN, MON_OUT",yes,yes
drift_rectangle,drift_rectangle.in,1,drift_rectangle.h5,"MON_IN, MON_OUT",yes,yes


rest mass check  drift_ellipse        deck implies 0.510999 MeV  vs REST_MASS_MEV = 0.510999  [ok]
rest mass check  drift_ellipse_np2    deck implies 0.510999 MeV  vs REST_MASS_MEV = 0.510999  [ok]
rest mass check  drift_noaperture     deck implies 0.510999 MeV  vs REST_MASS_MEV = 0.510999  [ok]
rest mass check  drift_rectangle      deck implies 0.510999 MeV  vs REST_MASS_MEV = 0.510999  [ok]


---
## 1. Stat browser

Pick one or more runs and one or more `.stat` columns. `x axis` switches between path length
`s`, time `t`, or both side by side. Ctrl/Cmd-click or shift-click to multi-select.

In [ ]:
X_LABEL = {"s": "s [m]", "t": "t [ns]"}


def plot_stats(cases_sel, cols_sel, xaxis, logy):
    if not cases_sel or not cols_sel:
        print("Select at least one run and one column.")
        return
    xs = ["s", "t"] if xaxis == "both" else [xaxis]
    data = {c: stat_of(CASES[c]) for c in cases_sel}
    units = data[cases_sel[0]][2]

    fig, axes = plt.subplots(len(cols_sel), len(xs),
                             figsize=(6.4 * len(xs), 3.0 * len(cols_sel)),
                             squeeze=False)
    for i, col in enumerate(cols_sel):
        for j, xc in enumerate(xs):
            ax = axes[i][j]
            for k, cname in enumerate(cases_sel):
                _, df, _ = data[cname]
                if col not in df.columns or xc not in df.columns:
                    continue
                ax.plot(df[xc], df[col], lw=1.4, label=cname,
                        color=CASE_COLORS[k % len(CASE_COLORS)])
            unit = units.get(col)
            ax.set_xlabel(X_LABEL[xc])
            ax.set_ylabel(f"{col} [{unit}]" if unit and unit != "1" else col)
            if logy:
                ax.set_yscale("log")
            if i == 0 and j == 0 and len(cases_sel) > 1:
                ax.legend(fontsize=8, framealpha=0.9)
    fig.tight_layout()
    plt.show()


_case_names = sorted(CASES)
w_cases = widgets.SelectMultiple(options=_case_names, value=tuple(_case_names),
                                 description="runs:", rows=min(8, max(3, len(_case_names))),
                                 layout=widgets.Layout(width="45%"))
w_cols = widgets.SelectMultiple(options=[], description="columns:", rows=8,
                                layout=widgets.Layout(width="45%"))
w_xaxis = widgets.ToggleButtons(options=["s", "t", "both"], value="s", description="x axis:")
w_logy = widgets.Checkbox(value=False, description="log y")


def _sync_columns(_=None):
    """Column list is the intersection over the selected runs, so a selection
    can never point at a column one of them lacks."""
    sel = list(w_cases.value)
    if not sel:
        w_cols.options = []
        return
    common = None
    for c in sel:
        cols = [x for x in stat_of(CASES[c])[1].columns if x not in ("t", "s")]
        common = cols if common is None else [x for x in common if x in cols]
    common = common or []
    keep = tuple(v for v in w_cols.value if v in common)
    w_cols.options = common
    w_cols.value = keep or (("numParticles",) if "numParticles" in common else
                            ((common[0],) if common else ()))


w_cases.observe(_sync_columns, names="value")
_sync_columns()

display(widgets.VBox([widgets.HBox([w_cases, w_cols]),
                      widgets.HBox([w_xaxis, w_logy])]),
        widgets.interactive_output(plot_stats, {"cases_sel": w_cases, "cols_sel": w_cols,
                                                "xaxis": w_xaxis, "logy": w_logy}))

Output()

---
## 2. Phase-space heatmaps

Monitor planes and bunch dumps as 2D histograms. Monitor files are read with **every** `Step#N`
group concatenated — one crossing is written as several groups, so reading only the first
under-reports it. Bunch dumps get a step slider instead, since there each group is a different
time.

**Two panels.** Panel A and panel B each pick their own run and file, so you can put two
monitors side by side — the same plane at two positions, or the same monitor from two runs.
Set panel B's file to *— none —* for a single panel.

**Fixed axes.** With `fixed axes` on (the default) both panels use the same `x ±` / `y ±`
limits *and the same bin edges and colour scale*, which is what makes the two densities
comparable by eye. Without it each panel auto-ranges to its own data and the comparison is
misleading. `fit to data` sets the limits from what is currently selected; they otherwise stay
put when you change file or run.

On the `x–y` plane the aperture declared in each panel's own deck is drawn on top.

In [ ]:
PLANES = {
    "x–y":     ("x", "y"),
    "x–x'":    ("x", "xp"),
    "y–y'":    ("y", "yp"),
    "px–py":   ("px", "py"),
    "z–x":     ("z", "x"),
}
AXIS_LABEL = {"x": "x [mm]", "y": "y [mm]", "z": "z [mm]",
              "xp": "x' [mrad]", "yp": "y' [mrad]",
              "px": "px [βγ]", "py": "py [βγ]"}


def _axis_values(d, key):
    """Values in plot units: positions in mm, divergences in mrad, momenta raw."""
    if key in ("x", "y", "z"):
        return d[key] * 1e3
    if key in ("xp", "yp"):
        return d["p" + key[0]] / d["pz"] * 1e3
    return d[key]


def _load_panel(case, path_str, step):
    """(data, path, label) for one panel. `data` is None when there is nothing to plot."""
    info = CASES[case]
    path = Path(path_str)
    is_bunch = info["bunch_h5"] is not None and path == info["bunch_h5"]
    d = read_step(path, step) if is_bunch else read_monitor(path)
    where = (f"Step#{step}" if is_bunch else
             f"{d['nsets']} Step group(s)" if d else "no records")
    return d, path, where


def nice_limit(v, margin=1.08):
    """Round a half-range up to 2 significant figures, with a little headroom.

    Rounding keeps the axis from twitching by a percent every time the
    selection changes, which is the point of fixing it in the first place.
    """
    v = float(v) * margin
    if not np.isfinite(v) or v <= 0:
        return 1.0
    e = np.floor(np.log10(v)) - 1
    return float(np.ceil(v / 10 ** e) * 10 ** e)


def plot_heatmaps(caseA, fileA, stepA, caseB, fileB, stepB,
                  plane, bins, logscale, fixed, xlim, ylim):
    if not fileA:
        print("Select a file for panel A.")
        return
    kx, ky = PLANES[plane]

    panels = []
    for case, fpath, step in ((caseA, fileA, stepA), (caseB, fileB, stepB)):
        if not fpath:
            continue
        d, path, where = _load_panel(case, fpath, step)
        if d is None:
            print(f"{case}/{path.name}: no particles.  A monitor nothing crosses "
                  "writes no records -- that is zero transmission, not an error.")
            continue
        for w in d.get("warnings", []):
            print("WARNING:", w)
        panels.append([case, path, where, d, _axis_values(d, kx), _axis_values(d, ky)])
    if not panels:
        return

    rng = [[-xlim, xlim], [-ylim, ylim]] if fixed else None

    # Bin both panels the same way and share one colour scale. Without this the
    # two densities are drawn on different scales and comparing them by eye is
    # misleading.
    vmax = 1
    for *_, u, v in panels:
        r = rng or [[u.min(), u.max()], [v.min(), v.max()]]
        vmax = max(vmax, np.histogram2d(u, v, bins=bins, range=r)[0].max())

    fig, axes = plt.subplots(1, len(panels), figsize=(6.0 * len(panels) + 1.4, 5.6),
                             squeeze=False, layout="constrained")
    mesh = None
    for ax, (case, path, where, d, u, v) in zip(axes[0], panels):
        kw = dict(bins=bins, range=rng, cmap="inferno", cmin=1)
        if logscale:
            kw["norm"] = LogNorm(vmin=1, vmax=vmax)     # norm and vmin/vmax are exclusive
        else:
            kw["vmin"], kw["vmax"] = 1, vmax
        mesh = ax.hist2d(u, v, **kw)[3]

        if plane == "x–y":
            ax.set_aspect("equal")      # 'box': keeps the limits, reshapes the axes
            apers = declared_apertures(CASES[case])
            for name, (shape, hx, hy) in apers.items():
                patch = (Ellipse((0, 0), 2 * hx * 1e3, 2 * hy * 1e3) if shape == "ell"
                         else Rectangle((-hx * 1e3, -hy * 1e3), 2 * hx * 1e3, 2 * hy * 1e3))
                patch.set(fill=False, ec="#39d3ff", lw=2, ls="--", zorder=5)
                ax.add_patch(patch)
                ax.plot([], [], color="#39d3ff", ls="--", lw=2,
                        label=f"{name}: {shape} {hx*1e3:g}×{hy*1e3:g} mm (half)")
            if apers:
                ax.legend(fontsize=8, loc="upper right", framealpha=0.9)

        if fixed:
            ax.set_xlim(-xlim, xlim)
            ax.set_ylim(-ylim, ylim)

        ex = geom_emittance(d["x"], d["px"] / d["pz"]) * 1e6
        ey = geom_emittance(d["y"], d["py"] / d["pz"]) * 1e6
        ax.set_xlabel(AXIS_LABEL[kx])
        ax.set_ylabel(AXIS_LABEL[ky])
        # Three short lines at a small size: one long title overruns the panel
        # and collides with the neighbouring colour bar.
        ax.set_title(f"{case} · {path.name} · {where}\n"
                     f"N={len(u)}   s={d['spos']:.4f} m\n"
                     f"rms x={d['x'].std()*1e3:.3f}  y={d['y'].std()*1e3:.3f} mm   "
                     f"ε {ex:.4g} / {ey:.4g} mm·mrad",
                     fontsize=9)
        ax.grid(alpha=0.15, color="w")

    # One bar for both panels -- they already share vmin/vmax, so a second one
    # would only repeat it and invite the reader to think the scales differ.
    fig.colorbar(mesh, ax=axes[0].tolist(), label="particles / bin", shrink=0.85)
    plt.show()


# ------------------------------------------------------------------- controls --
NO_FILE = ("— none —", "")


def _file_options(case, allow_none=False):
    info = CASES[case]
    files = list(info["monitors"]) + ([info["bunch_h5"]] if info["bunch_h5"] else [])
    opts = [(p.name, str(p)) for p in files]
    return ([NO_FILE] + opts) if allow_none else opts


def _preferred(opts):
    """Default to the downstream monitor -- the interesting plane in a scraping run."""
    for label, value in opts:
        if value and "OUT" in label.upper():
            return value
    return opts[0][1] if opts else ""


def _make_panel(default_case, allow_none, tag):
    case = widgets.Dropdown(options=_case_names, value=default_case,
                            description=f"{tag} run:", layout=widgets.Layout(width="95%"))
    opts = _file_options(default_case, allow_none)
    file = widgets.Dropdown(options=opts, value=_preferred(opts),
                            description=f"{tag} file:", layout=widgets.Layout(width="95%"))
    step = widgets.IntSlider(description=f"{tag} step:", value=0, min=0, max=0,
                             continuous_update=False, disabled=True,
                             layout=widgets.Layout(width="95%"))

    def sync_files(_=None):
        keep = Path(file.value).name if file.value else None
        opts = _file_options(case.value, allow_none)
        file.options = opts
        match = next((v for l, v in opts if l == keep), None)
        file.value = match if match is not None else _preferred(opts)

    def sync_steps(_=None):
        info = CASES[case.value]
        if not file.value or info["bunch_h5"] is None or Path(file.value) != info["bunch_h5"]:
            step.disabled = True
            step.min = step.max = step.value = 0
            return
        steps = h5_steps(file.value)
        step.disabled = False
        step.min, step.max = (min(steps), max(steps)) if steps else (0, 0)
        step.value = step.max

    case.observe(lambda c: (sync_files(), sync_steps()), names="value")
    file.observe(sync_steps, names="value")
    sync_steps()
    return case, file, step


hA_case, hA_file, hA_step = _make_panel(_case_names[0], False, "A")
hB_case, hB_file, hB_step = _make_panel(_case_names[min(1, len(_case_names) - 1)], True, "B")

h_plane = widgets.Dropdown(options=list(PLANES), value="x–y", description="plane:")
h_bins = widgets.IntSlider(description="bins:", value=50, min=10, max=200, step=10,
                           continuous_update=False)
h_log = widgets.Checkbox(value=False, description="log counts")
h_fixed = widgets.Checkbox(value=True, description="fixed axes")
h_xlim = widgets.FloatText(value=12.0, step=1.0, description="x ±:",
                           layout=widgets.Layout(width="180px"))
h_ylim = widgets.FloatText(value=12.0, step=1.0, description="y ±:",
                           layout=widgets.Layout(width="180px"))
h_fit = widgets.Button(description="fit to data", icon="arrows-alt")


def _fit_limits(_=None):
    """Set the fixed limits from whatever is selected right now."""
    kx, ky = PLANES[h_plane.value]
    us, vs = [], []
    for case, file, step in ((hA_case, hA_file, hA_step), (hB_case, hB_file, hB_step)):
        if not file.value:
            continue
        d, _, _ = _load_panel(case.value, file.value, step.value)
        if d is None:
            continue
        us.append(np.abs(_axis_values(d, kx)).max())
        vs.append(np.abs(_axis_values(d, ky)).max())
    if us:
        h_xlim.value = nice_limit(max(us))
    if vs:
        h_ylim.value = nice_limit(max(vs))


h_fit.on_click(_fit_limits)
# Units change completely between planes (mm vs mrad vs beta*gamma), so a limit
# carried over from another plane would be meaningless -- refit on every change.
h_plane.observe(_fit_limits, names="value")
_fit_limits()

display(widgets.VBox([
    widgets.HBox([widgets.VBox([hA_case, hA_file, hA_step],
                               layout=widgets.Layout(width="50%")),
                  widgets.VBox([hB_case, hB_file, hB_step],
                               layout=widgets.Layout(width="50%"))]),
    widgets.HBox([h_plane, h_bins, h_log]),
    widgets.HBox([h_fixed, h_xlim, h_ylim, h_fit]),
]), widgets.interactive_output(plot_heatmaps, {
    "caseA": hA_case, "fileA": hA_file, "stepA": hA_step,
    "caseB": hB_case, "fileB": hB_file, "stepB": hB_step,
    "plane": h_plane, "bins": h_bins, "logscale": h_log,
    "fixed": h_fixed, "xlim": h_xlim, "ylim": h_ylim}))

Output()

---
## 3. ParaView export

Writes into `<run>/output/paraview/`:

| file | what |
| --- | --- |
| `bunch_lab.pvd` + `bunch_lab_stepNNNNN.vtp` | bunch per step, **lab frame** |
| `bunch_comoving.pvd` + `bunch_comoving_stepNNNNN.vtp` | bunch per step, as stored in the `.h5` |
| `lattice_elements.vtp` | aperture tubes, coloured by `element_type` |
| `lattice_reforbit.vtp` | design orbit polyline |

The `.h5` holds co-moving coordinates: `x`, `y`, `z` are relative to the reference particle.
The lab frame applies `r_lab = RefPartR + Q(s)·r_local`, where `Q(s)` is built by parallel-
transporting the reference momentum direction along `data/*_DesignPath.dat`. Only the lab-frame
files share a frame with the lattice geometry — load those together.

Point arrays on every particle: `P` (momentum vector, rotated with the frame), `Pmag` (βγ),
`Ekin_MeV` (from `REST_MASS_MEV`), plus every other per-particle dataset in the file.

In [ ]:
e_cases = widgets.SelectMultiple(options=_case_names, value=tuple(_case_names[:1]),
                                 description="runs:", rows=min(8, max(3, len(_case_names))),
                                 layout=widgets.Layout(width="45%"))
e_lab = widgets.Checkbox(value=True, description="lab frame")
e_com = widgets.Checkbox(value=True, description="co-moving frame")
e_geom = widgets.Checkbox(value=True, description="lattice geometry")
e_stride = widgets.IntSlider(description="step stride:", value=1, min=1, max=10,
                             continuous_update=False)
e_button = widgets.Button(description="Export to ParaView", button_style="primary",
                          icon="download")
e_out = widgets.Output()


def _do_export(_):
    e_out.clear_output()
    with e_out:
        frames = tuple(f for f, on in (("lab", e_lab.value), ("comoving", e_com.value)) if on)
        if not frames and not e_geom.value:
            print("Nothing selected.")
            return
        for name in e_cases.value:
            export_paraview(CASES[name], frames=frames, stride=e_stride.value,
                            geometry=e_geom.value)
        print("\ndone.")


e_button.on_click(_do_export)
display(widgets.VBox([e_cases,
                      widgets.HBox([e_lab, e_com, e_geom]),
                      widgets.HBox([e_stride, e_button]), e_out]))

**In ParaView**: open `bunch_lab.pvd`, `lattice_elements.vtp` and `lattice_reforbit.vtp`
together. Set the elements' representation to Surface and drop Opacity to ~0.25 so the
particles show through, and colour them by `element_type`
(0 DRIFT, 1 SOLENOID, 2 DIPOLE, 3 QUADRUPOLE, 4 MONITOR, 5 other). Colour the bunch by
`Ekin_MeV` or `Pmag`; add a Glyph filter oriented by `P` for direction. Press play to animate
over the step times.

---
## Self-test

Runs the three sections without the widgets and checks the results, so executing this notebook
end to end (`jupyter nbconvert --execute`) actually proves something. Set `RUN_SELFTEST = False`
to skip.

In [ ]:
RUN_SELFTEST = True

if RUN_SELFTEST:
    import tempfile

    failures = []

    def check(label, ok, note=""):
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}" + (f"  ({note})" if note else ""))
        if not ok:
            failures.append(label)

    print("stat reading")
    for name, c in CASES.items():
        _, df, _ = stat_of(c)
        check(f"{name}: stat has s/t/numParticles and rows",
              {"s", "t", "numParticles"} <= set(df.columns) and len(df) > 0,
              f"{len(df)} rows, final N={df['numParticles'].iloc[-1]:.0f}")

    print("\nmonitor reading")
    for name, c in CASES.items():
        for m in c["monitors"]:
            d = read_monitor(m)
            check(f"{name}/{m.stem}: readable", d is not None and len(d["x"]) > 0,
                  f"N={len(d['x'])}, {d['nsets']} step group(s)" if d else "no records")

    print("\ndeck aperture parsing")
    for name, c in CASES.items():
        if not c["deck"]:
            continue
        # A deck that sets APERTURE must yield one, otherwise the parser is
        # silently returning nothing and the check below would vacuously pass.
        # Match the attribute, not the bare word -- `Title, string="drift_noaperture"`
        # contains it as a substring.
        text = strip_comments(c["deck"].read_text())
        mentions = bool(re.search(r"\b(APERTURE|HAPERT)\s*=", text, re.IGNORECASE))
        apers = declared_apertures(c)
        check(f"{name}: deck sets APERTURE -> parsed",
              bool(apers) == mentions,
              ", ".join(f"{n}={a[0]} {a[1]*1e3:g}x{a[2]*1e3:g} mm" for n, a in apers.items())
              or "none declared")

    print("\nsurvivors inside the declared aperture (x-y at the last monitor)")
    for name, c in CASES.items():
        apers = declared_apertures(c)
        mon = next((m for m in c["monitors"] if m.stem.upper().endswith("OUT")), None)
        if not apers or mon is None:
            print(f"  [skip] {name}: no declared aperture or no MON_OUT")
            continue
        d = read_monitor(mon)
        shape, hx, hy = next(iter(apers.values()))
        r = ((d["x"] / hx) ** 2 + (d["y"] / hy) ** 2 if shape == "ell"
             else np.maximum(np.abs(d["x"]) / hx, np.abs(d["y"]) / hy))
        check(f"{name}: no survivor outside {shape}", float(r.max()) < 1.0,
              f"max normalised radius {r.max():.6f}")

    print("\nheatmap rendering")
    _names = sorted(CASES)
    _a, _b = _names[0], _names[min(1, len(_names) - 1)]
    _ma, _mb = str(CASES[_a]["monitors"][-1]), str(CASES[_b]["monitors"][-1])
    for _label, _fb in (("one panel", ""), ("two panels", _mb)):
        try:
            plot_heatmaps(_a, _ma, 0, _b, _fb, 0, "x–y", 40, False, True, 12.0, 12.0)
            ok, note = True, ""
        except Exception as exc:
            ok, note = False, repr(exc)
        check(f"{_label} renders", ok, note)
    check("nice_limit rounds up with headroom",
          nice_limit(10.0) >= 10.0 and nice_limit(10.0) < 12.0,
          f"nice_limit(10) = {nice_limit(10.0)}")

    print("\nParaView export (straight-line lattice: the lab rotation is the identity,")
    print("so lab x,y must equal local x,y and lab z must equal local z + SPOS)")
    name = next(iter(CASES))
    c = CASES[name]
    if c["bunch_h5"] is not None:
        with tempfile.TemporaryDirectory() as tmp:
            pvd = export_bunch(c, "lab", tmp, log=lambda *a: None)
            geo = export_geometry(c, tmp, log=lambda *a: None)
            check(f"{name}: bunch_lab.pvd written", pvd is not None and Path(pvd).is_file())
            vtps = sorted(Path(tmp).glob("bunch_lab_step*.vtp"))
            check(f"{name}: one .vtp per step",
                  len(vtps) == len(h5_steps(c["bunch_h5"])),
                  f"{len(vtps)} files")
            check(f"{name}: lattice files written", len(geo) == 2,
                  ", ".join(p.name for p in geo))

            import vtk
            from vtk.util.numpy_support import vtk_to_numpy
            rd = vtk.vtkXMLPolyDataReader()
            rd.SetFileName(str(vtps[0])); rd.Update()
            poly = rd.GetOutput()
            lab = vtk_to_numpy(poly.GetPoints().GetData())
            ekin = vtk_to_numpy(poly.GetPointData().GetArray("Ekin_MeV"))
            step0 = h5_steps(c["bunch_h5"])[0]
            loc = read_step(c["bunch_h5"], step0)
            dxy = max(np.abs(lab[:, 0] - loc["x"]).max(), np.abs(lab[:, 1] - loc["y"]).max())
            dz = np.abs(lab[:, 2] - (loc["z"] + loc["spos"])).max()
            check(f"{name}: lab x,y == local x,y", dxy < 1e-12, f"max |d| = {dxy:.2e} m")
            check(f"{name}: lab z == local z + SPOS", dz < 1e-9, f"max |d| = {dz:.2e} m")
            check(f"{name}: Ekin from REST_MASS_MEV is sane",
                  bool(np.isfinite(ekin).all()) and ekin.mean() > 0,
                  f"mean {ekin.mean():.3f} MeV")

    print()
    if failures:
        raise AssertionError(f"{len(failures)} self-test check(s) failed: {failures}")
    print("self-test: all checks passed.")